# 03 — Divisão treino / validação (dados mockados)

Objetivo: dividir o dataset em conjuntos de treino (80%) e validação (20%), de forma estratificada (mantendo a proporção 50/50 entre as classes "correto" e "erro" em cada conjunto).

O conjunto de TESTE final não é fatiado a partir daqui -- ele já existe separadamente em `dataset_teste_externo/`, seguindo o planejamento original do projeto (peças extras, nunca vistas durante treino/validação).

O resultado deste notebook é salvo em `dataset/train.csv` e `dataset/val.csv`, listando explicitamente quais arquivos pertencem a cada conjunto -- isso garante reprodutibilidade: qualquer pessoa (ou você mesma, no futuro) pode ver exatamente quais imagens foram usadas em cada etapa, sem depender de rodar o split de novo e torcer para dar
o mesmo resultado.

## Imports

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Reprodutibilidade: mesmo split sempre que o notebook for executado.
SEED = 42

## Carregamento do CSV e split estratificado

In [5]:
# Le o CSV gerado pelo scripts/gerar_mock.py, que lista cada arquivo
# do dataset principal (100 imagens: 50 correto + 50 erro) com sua
# respectiva classe e tipo de erro.
caminho_csv = "../dataset/labels_mock.csv"
df = pd.read_csv(caminho_csv)

print(f"Total de registros no CSV: {len(df)}")
print(f"Distribuicao por classe:\n{df['Classe'].value_counts()}")

# train_test_split com stratify=df['Classe']: garante que a PROPORCAO
# de cada classe (correto/erro) seja preservada tanto no conjunto de
# treino quanto no de validacao. Sem isso, um split aleatorio simples
# poderia, por azar, concentrar mais "erro" no treino e mais "correto"
# na validacao (ou vice-versa) -- especialmente arriscado com um
# dataset pequeno como o nosso (100 imagens).
#
# test_size=0.2 -> 20% para validacao, 80% para treino (decisao ja
# fechada: dataset/ vira treino+validacao; o teste final fica a cargo
# do holdout externo em dataset_teste_externo/).
df_treino, df_validacao = train_test_split(
    df,
    test_size=0.2,
    stratify=df["Classe"],
    random_state=SEED,
)

print(f"\nTreino: {len(df_treino)} imagens")
print(df_treino["Classe"].value_counts())

print(f"\nValidacao: {len(df_validacao)} imagens")
print(df_validacao["Classe"].value_counts())

Total de registros no CSV: 100
Distribuicao por classe:
Classe
correto    50
erro       50
Name: count, dtype: int64

Treino: 80 imagens
Classe
correto    40
erro       40
Name: count, dtype: int64

Validacao: 20 imagens
Classe
erro       10
correto    10
Name: count, dtype: int64


## Salvar o resultado do split em CSV

In [6]:
# Salva cada conjunto em um CSV separado, dentro de dataset/ -- assim
# fica documentado e reprodutivel exatamente quais arquivos pertencem
# a treino e quais pertencem a validacao, sem depender de rodar o
# split de novo (e torcer para dar o mesmo resultado).
#
# index=False: evita que o pandas escreva uma coluna extra com o
# indice numerico original do DataFrame (que nao tem nenhum valor
# informativo aqui, so poluiria o CSV).
caminho_train = "../dataset/train.csv"
caminho_val = "../dataset/val.csv"

df_treino.to_csv(caminho_train, index=False)
df_validacao.to_csv(caminho_val, index=False)

print(f"train.csv salvo em: {caminho_train} ({len(df_treino)} registros)")
print(f"val.csv salvo em: {caminho_val} ({len(df_validacao)} registros)")

train.csv salvo em: ../dataset/train.csv (80 registros)
val.csv salvo em: ../dataset/val.csv (20 registros)
